In [3]:
!nvidia-smi


Fri Nov  6 10:02:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P0             26W /   70W |     425MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch
import time

size = 5000
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
c_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start
print(f"CPU: {cpu_time:.3f}s")

if torch.cuda.is_available():
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")
    torch.cuda.synchronize()
    start = time.time()
    c_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()
    gpu_time = time.time() - start
    print(f"GPU: {gpu_time:.3f}s")
    print(f"Speedup: {cpu_time / gpu_time:.0f}x")


CPU: 2.015s
GPU: 0.088s
Speedup: 23x


In [5]:
import torch

total = torch.cuda.get_device_properties(0).total_memory
print(f"Total GPU memory: {total / 1e9:.1f} GB")

max_params = total / 2
print(f"Ceiling with no margin (fp16): {max_params / 1e9:.1f} billion parameters")

usable = total * 0.8
print(f"With 20% kept aside for activations + CUDA: {usable / 2 / 1e9:.1f} billion parameters")


Total GPU memory: 15.6 GB
Ceiling with no margin (fp16): 7.8 billion parameters
With 20% kept aside for activations + CUDA: 6.3 billion parameters
